# Exp 7 — Answering the reviewer: batch size, a second GPU, and pose estimation

An AI pre-review raised three questions; this notebook answers them. It runs on Kaggle's **T4**, and Part C repeats key runs on the **CPU** of the same session as a second, completely different hardware back end.

**Part A — Batch size.** Do 2, 4, 16 or 32 images per batch behave like 8? We sweep 1 → 2 → 4 → 8 → 16 → 32 on 1,000 COCO val images for DETR, Faster R-CNN and YOLO11m, and run our fix at batch 32. We expect the share of changed images to grow with batch size, because the chance that an image is *not* the largest in its batch grows. Among padded images, the rate should stay roughly constant (the "switch").

**Part B — Pose estimation.** Keypoint R-CNN (torchvision) on all 5,000 COCO val images: batch 1, random batch 8, our fix, and exact-shape buckets. We report box and keypoint AP, the share of images whose detections change, and the share whose *poses* change (a matched person's keypoints move by more than 5% of the box size).

**Part C — Second hardware back end (CPU).** On 200 images, DETR, Faster R-CNN and YOLO11m run at batch 1 and in random batches of 8 (plus our fix), once on the T4 and once on the CPU, with identical batches. Does the effect, and the fix, replicate on the CPU? And do batch-1 outputs agree between the two devices?

**Claims (fixed before running):**
- **R1 (batch size):** for each padding-based model, the share of changed images grows with batch size from 2 to 32, and among *padded* images it stays within ±10 points across sizes. Our fix at batch 32 changes ≤ 0.5% of images.
- **R2 (pose):** normal batching changes ≥ 10% of images' poses for Keypoint R-CNN; our fix changes ≤ 0.5%, with |Δ keypoint AP| ≤ 0.05.
- **R3 (hardware):** on the CPU, batch-8 change rates are within 5 points of the T4 rates on the same images and batches, and our fix stays ≤ 0.5% on both devices.

**Kaggle:** Accelerator **GPU T4 x2** (or T4), Internet **ON**, nothing to attach. About 3.5–4.5 hours (the CPU part is slow by nature); run as **Save Version → Save & Run All (Commit)**.

In [ ]:
!pip install -q "transformers==5.17.0" pycocotools timm "ultralytics==8.4.172"

In [ ]:
import os
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")   # less memory fragmentation
import io, sys, glob, json, time, math, random, zipfile, platform, datetime, contextlib, urllib.request, dataclasses, gc
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch, torchvision
import torch.nn.functional as F
import torch.nn as nn
import inspect
import xml.etree.ElementTree as ET
from collections import deque
from pycocotools import mask as mask_utils
import matplotlib.pyplot as plt
from PIL import Image
import transformers
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

OUT = "/kaggle/working/exp7"
os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

ENV = {
    "run_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps(ENV, indent=2))
if ENV["gpu"] is None:
    print("\n⚠⚠ NO GPU. Stop, set Accelerator to GPU in the right panel, and Run All again. ⚠⚠")

def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

## 1. Configuration

In [ ]:
CFG = {
    "seed": 0,
    "batch_size": 8,
    "sweep_models": ["detr_r50", "frcnn_v2", "yolo11m"],
    "sweep_batches": [2, 4, 8, 16, 32],
    "n_sweep": 1000,
    "pose_model": "keypointrcnn",
    "pose_change": 0.05,            # mean keypoint displacement / sqrt(box area) counted as a changed pose
    "yolo_imgsz": 640, "yolo_conf": 0.001, "yolo_iou": 0.7,
    "micro_batch": {},
    "flip_thr_per_model": {},
    "flip_score_thr": 0.5,
    "flip_iou": 0.9,
    "coco_root": "/kaggle/input",
    "coco_download_dir": "/kaggle/working/coco",
    "cpu_models": ["detr_r50", "frcnn_v2", "yolo11m"],
    "n_cpu": 200,
    "claims": {"R1_band": 0.10, "R1_fix": 0.005, "R2_normal": 0.10, "R2_fix": 0.005, "R2_dap": 0.05, "R3_tol": 0.05, "R3_fix": 0.005},
}
AMP = False
RUN_PARTS = {"A": True, "B": True, "C": True}
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
torch.set_num_threads(os.cpu_count() or 4)
print("GPU:", ENV["gpu"], "| CPU threads:", torch.get_num_threads())

## 2. Data: COCO val2017 (boxes and keypoints) and a 1,000-image subset

In [ ]:
class LazyImages:
    """Images are decoded from disk only when needed."""
    def __init__(self, paths):
        self.paths = list(paths)
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, k):
        if isinstance(k, slice):
            return [self[i] for i in range(*k.indices(len(self)))]
        with Image.open(self.paths[k]) as im:
            return im.convert("RGB")

# ---------------- COCO val2017 (all 5,000 images) ----------------
def find_attached_coco(root):
    anns = glob.glob(os.path.join(root, "**", "instances_val2017.json"), recursive=True)
    probe = glob.glob(os.path.join(root, "**", "000000000139.jpg"), recursive=True)
    return (anns[0], os.path.dirname(probe[0])) if anns and probe else (None, None)

ANN, COCO_IMG_DIR = find_attached_coco(CFG["coco_root"])
if ANN is None:
    D = CFG["coco_download_dir"]; os.makedirs(D, exist_ok=True)
    ANN, COCO_IMG_DIR = os.path.join(D, "instances_val2017.json"), os.path.join(D, "val2017")
    if not os.path.exists(ANN):
        zp = os.path.join(D, "ann.zip"); print("downloading COCO annotations (~250 MB) ...")
        urllib.request.urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", zp)
        with zipfile.ZipFile(zp) as z, open(ANN, "wb") as f:
            f.write(z.read("annotations/instances_val2017.json"))
        os.remove(zp)
    if not (os.path.isdir(COCO_IMG_DIR) and len(os.listdir(COCO_IMG_DIR)) >= 5000):
        zp = os.path.join(D, "val2017.zip"); print("downloading COCO val2017 images (~800 MB) ...")
        urllib.request.urlretrieve("http://images.cocodataset.org/zips/val2017.zip", zp)
        with zipfile.ZipFile(zp) as z:
            z.extractall(D)
        os.remove(zp)
with contextlib.redirect_stdout(io.StringIO()):
    COCO_GT = COCO(ANN)
NAME2CAT = {c["name"].strip().lower(): c["id"] for c in COCO_GT.loadCats(COCO_GT.getCatIds())}
_ids = sorted(COCO_GT.getImgIds()); _info = COCO_GT.loadImgs(_ids)
DATASETS = {"coco": dict(GT=COCO_GT, IDS=_ids, INFO=_info,
                         IMGS=LazyImages(os.path.join(COCO_IMG_DIR, x["file_name"]) for x in _info))}
GT = COCO_GT   # default dataset until use_dataset() switches
print(f"COCO val2017: {len(_ids)} images")


# keypoint annotations (same zip as the boxes)
KP_ANN = os.path.join(os.path.dirname(ANN), "person_keypoints_val2017.json")
if not os.path.exists(KP_ANN):
    zp = os.path.join(CFG["coco_download_dir"], "ann2.zip"); os.makedirs(CFG["coco_download_dir"], exist_ok=True)
    print("downloading COCO keypoint annotations ...")
    urllib.request.urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", zp)
    with zipfile.ZipFile(zp) as z, open(KP_ANN, "wb") as f:
        f.write(z.read("annotations/person_keypoints_val2017.json"))
    os.remove(zp)
with contextlib.redirect_stdout(io.StringIO()):
    KP_GT = COCO(KP_ANN)
# 1,000-image subset for the batch-size sweep
_sub = sorted(random.Random(CFG["seed"]).sample(range(len(_ids)), CFG["n_sweep"]))
DATASETS["coco1k"] = dict(GT=COCO_GT, IDS=[_ids[i] for i in _sub], INFO=[_info[i] for i in _sub],
                          IMGS=LazyImages(DATASETS["coco"]["IMGS"].paths[i] for i in _sub))
DATASETS["cpu200"] = dict(GT=COCO_GT, IDS=[_ids[i] for i in _sub[:CFG["n_cpu"]]], INFO=[_info[i] for i in _sub[:CFG["n_cpu"]]],
                          IMGS=LazyImages(DATASETS["coco"]["IMGS"].paths[i] for i in _sub[:CFG["n_cpu"]]))
print("sweep subset:", len(_sub), "images; CPU subset:", CFG["n_cpu"])

## 3. The `self` fix

In [ ]:
class Fix:
    COMPONENTS = ("conv", "norm", "own", "exactmask")
    def __init__(self):
        for c in self.COMPONENTS:
            setattr(self, c, False)
        self.valid, self.canvas, self.trivial, self._cache = None, None, True, {}

    def set(self, components):
        for c in self.COMPONENTS:
            setattr(self, c, c in components)

    def any_on(self):
        return any(getattr(self, c) for c in self.COMPONENTS)

    def begin(self, valid, canvas):
        self.valid = [tuple(int(v) for v in s) for s in valid]
        self.canvas = tuple(int(c) for c in canvas)
        self.trivial = all(v == self.canvas for v in self.valid)

    def end(self):
        self.valid = None

    def active(self):
        return self.valid is not None and not self.trivial

    def applies(self, x):
        return self.active() and x.dim() == 4 and x.shape[0] == len(self.valid)

    @staticmethod
    def _stride(n, N):
        for s in (1, 2, 4, 8, 16, 32, 64, 128, 256):
            if math.ceil(N / s) == n:
                return s
        return N / n

    def extents(self, H, W):
        """each image's valid (rows, cols) at a feature map of size H x W"""
        sh, sw = self._stride(H, self.canvas[0]), self._stride(W, self.canvas[1])
        return [(math.ceil(h / sh), math.ceil(w / sw)) for h, w in self.valid]

    def mask(self, x):
        H, W = x.shape[-2:]
        key = (H, W, tuple(self.valid), self.canvas, x.device, x.dtype)
        if key not in self._cache:
            m = torch.zeros(len(self.valid), 1, H, W, device=x.device, dtype=x.dtype)
            for i, (h, w) in enumerate(self.extents(H, W)):
                m[i, :, :h, :w] = 1
            if len(self._cache) > 64:
                self._cache.clear()
            self._cache[key] = m
        return self._cache[key]

def _mask_conv_inputs(root, fix):
    """Zero the padding at the input of every conv / max-pool: each layer then sees exactly the zeros its
    batch-1 run sees past the canvas edge (activations are >= 0 after ReLU, so max-pool is unaffected)."""
    def pre(mod, inp):
        x = inp[0]
        if fix.conv and fix.applies(x):
            return (x * fix.mask(x),) + tuple(inp[1:])
    for m in root.modules():
        if isinstance(m, (nn.Conv2d, nn.MaxPool2d)):
            m.register_forward_pre_hook(pre)

def _masked_groupnorm(root, fix):
    """GroupNorm statistics over each image's own valid area only."""
    n = 0
    for mod in root.modules():
        if isinstance(mod, nn.GroupNorm):
            orig = mod.forward
            def fwd(x, mod=mod, orig=orig):
                if not (fix.norm and fix.applies(x)):
                    return orig(x)
                m = fix.mask(x).float(); B, C, H, W = x.shape; G = mod.num_groups
                xg, mg = x.float().reshape(B, G, C // G, H, W), m.reshape(B, 1, 1, H, W)
                cnt = mg.sum(dim=(2, 3, 4), keepdim=True) * (C // G)
                mean = (xg * mg).sum(dim=(2, 3, 4), keepdim=True) / cnt
                var = (((xg - mean) ** 2) * mg).sum(dim=(2, 3, 4), keepdim=True) / cnt
                y = ((xg - mean) / torch.sqrt(var + mod.eps)).reshape(B, C, H, W)
                if mod.affine:
                    y = y * mod.weight.float()[None, :, None, None] + mod.bias.float()[None, :, None, None]
                return y.to(x.dtype)
            mod.forward = fwd; n += 1
    return n

def _suppress_padding_scores(conv, fix):
    """Classification logits at positions that do not exist in the image's batch-1 run -> -1e4."""
    def hook(mod, inp, out):
        if fix.conv and fix.applies(out):
            return out.masked_fill(fix.mask(out) == 0, -1e4)
    conv.register_forward_hook(hook)

def _own_anchors(ag, fix):
    """Each image gets the anchors its own batch-1 canvas would give: torchvision spaces anchors at
    canvas // grid, so we use the image's own canvas and grid. Anchors at positions that do not exist in
    its batch-1 run become zero-size boxes, which torchvision removes."""
    orig = ag.forward
    def fwd(image_list, feature_maps):
        if not (fix.own and fix.active()):
            return orig(image_list, feature_maps)
        grid_sizes = [list(f.shape[-2:]) for f in feature_maps]
        Hc, Wc = image_list.tensors.shape[-2:]
        dtype, device = feature_maps[0].dtype, feature_maps[0].device
        ag.set_cell_anchors(dtype, device)
        n_cell = [ca.shape[0] for ca in ag.cell_anchors]
        out = []
        for ch, cw in fix.valid:
            strides, keep = [], []
            for (gh, gw), a in zip(grid_sizes, n_cell):
                sh, sw = 2 ** round(math.log2(Hc / gh)), 2 ** round(math.log2(Wc / gw))
                oh, ow = math.ceil(ch / sh), math.ceil(cw / sw)
                strides.append([torch.tensor(ch // oh, dtype=torch.int64, device=device),
                                torch.tensor(cw // ow, dtype=torch.int64, device=device)])
                k = torch.zeros(gh, gw, dtype=dtype, device=device); k[:oh, :ow] = 1
                keep.append(k.reshape(-1).repeat_interleave(a))
            anc = ag.grid_anchors(grid_sizes, strides)
            out.append(torch.cat([x * k[:, None] for x, k in zip(anc, keep)]))
        return out
    ag.forward = fwd

def install_fix_tv(model, fix):
    heads = [model.rpn.head] if hasattr(model, "rpn") else [model.head]
    for root in [model.backbone] + heads:
        _mask_conv_inputs(root, fix)
    has_roi = hasattr(model, "roi_heads")
    def zero_outputs(mod, inp, out):
        if not (fix.conv and fix.active()):
            return
        new = type(out)()
        for k, v in out.items():
            v = v * fix.mask(v)
            if fix.own and has_roi:
                # RoIAlign at batch 1 clamps samples at the canvas edge to the last row/column; copying that
                # row/column one step outward reproduces this inside a bigger canvas.
                H, W = v.shape[-2:]
                for i, (hl, wl) in enumerate(fix.extents(H, W)):
                    if wl < W:
                        v[i, :, :hl, wl] = v[i, :, :hl, wl - 1]
                    if hl < H:
                        v[i, :, hl, :min(wl + 1, W)] = v[i, :, hl - 1, :min(wl + 1, W)]
            new[k] = v
        return new
    model.backbone.register_forward_hook(zero_outputs)
    g = sum(_masked_groupnorm(h, fix) for h in heads) + _masked_groupnorm(model.backbone, fix)
    if hasattr(model, "rpn"):
        _own_anchors(model.rpn.anchor_generator, fix); _suppress_padding_scores(model.rpn.head.cls_logits, fix)
    else:
        _own_anchors(model.anchor_generator, fix); _suppress_padding_scores(model.head.classification_head.cls_logits, fix)
    fix.SELF = ("conv", "norm", "own") if g else ("conv", "own")

def install_fix_hf(model, fix):
    enc = model.model.backbone
    _mask_conv_inputs(enc, fix)
    orig = enc.forward
    def fwd(pixel_values, pixel_mask):
        out = orig(pixel_values, pixel_mask)
        if not (fix.exactmask and fix.active()):
            return out
        # the transformer's padding mask with exact extents (the library shrinks it with nearest-neighbour rounding)
        return [(fm, fix.mask(fm)[:, 0].bool()) for fm, _ in out]
    enc.forward = fwd
    fix.SELF = ("conv", "exactmask")

## 4. Detectors (incl. Ultralytics YOLO and Keypoint R-CNN)

In [ ]:
ALIASES = {"motorbike": "motorcycle", "aeroplane": "airplane", "sofa": "couch", "tvmonitor": "tv", "tv monitor": "tv",
           "diningtable": "dining table", "pottedplant": "potted plant", "hair dryer": "hair drier",
           "sports ball": "sports ball", "cellphone": "cell phone", "mobile phone": "cell phone"}
def _norm(s):
    s = " ".join(str(s).lower().replace("_", " ").replace("-", " ").split())
    return ALIASES.get(s, s)
def names_to_cat(names, label=""):
    m, miss = {}, []
    for k, v in names.items():
        c = NAME2CAT.get(_norm(v))
        if c is not None:
            m[int(k)] = c
        elif _norm(v) not in ("n/a", "__background__", "background", "no object"):
            miss.append(v)
    if miss:
        print(f"  unmapped class names in {label}: {miss}")
    return m

def load_hf_model(cls, repo, revision=None):
    try:
        return cls.from_pretrained(repo, revision=revision)
    except Exception as ex:
        if "alidation" not in f"{type(ex).__name__} {ex}":
            raise
        from huggingface_hub import hf_hub_download
        raw = json.load(open(hf_hub_download(repo, "config.json", revision=revision)))
        defaults = {f.name: f.default for f in dataclasses.fields(cls.config_class) if f.default is not dataclasses.MISSING}
        dropped = [k for k, v in raw.items() if v is None and defaults.get(k) is not None]
        print(f"  (config fix for {repo}: replaced null fields {dropped} with library defaults)")
        raw = {k: v for k, v in raw.items() if k not in dropped}
        return cls.from_pretrained(repo, revision=revision, config=cls.config_class.from_dict(raw))

def roundup(x, d):
    return int(math.ceil(x / d) * d)

def read_csv(path):
    try:
        return pd.read_csv(path)
    except (FileNotFoundError, pd.errors.EmptyDataError):
        return pd.DataFrame()

def _pack(boxes, scores, labels, lab2cat):
    lab = labels.cpu().numpy()
    keep = np.array([l in lab2cat for l in lab], bool)
    return {"boxes": boxes.float().cpu().numpy()[keep], "scores": scores.float().cpu().numpy()[keep],
            "labels": np.array([lab2cat[l] for l in lab[keep]], dtype=int)}

class MicroBatch:
    """If a batch does not fit in GPU memory, split it into smaller pieces that are all padded to the canvas the
    full batch would have had. Images in these models never interact except through padding, so this gives the
    same result as the full batch (verified for DETR by the canvas self-test, and for every Part-B model by B0)."""
    micro = None   # largest piece size known to fit; None = no limit found yet

    def batch_canvas(self, imgs):
        sizes = [self.input_size(im) for im in imgs]
        return (roundup(max(h for h, _ in sizes), self.div), roundup(max(w for _, w in sizes), self.div))

    def run(self, imgs, canvas=None):
        if self.fixed_size:
            return self._run(imgs, canvas)
        if self.micro is not None and len(imgs) > self.micro:
            return self._split(imgs, canvas)
        try:
            return self._run(imgs, canvas)
        except torch.cuda.OutOfMemoryError:
            if len(imgs) == 1:
                raise
        # We are now OUTSIDE the except block, so the failed attempt (and the GPU memory it holds) is released
        # before retrying. Retrying inside the except block keeps that memory alive.
        free_mem()
        self.micro = max(1, len(imgs) // 2)
        print(f"  [{self.name}] batch of {len(imgs)} does not fit in GPU memory → running pieces of {self.micro} "
              f"on the full batch's canvas (equivalent result)")
        return self._split(imgs, canvas)

    def _split(self, imgs, canvas):
        canvas = canvas if canvas is not None else self.batch_canvas(imgs)
        step = self.micro                      # fixed for this call, even if a piece lowers self.micro further
        R, P = [], []
        for k in range(0, len(imgs), step):
            r, p = self.run(imgs[k:k + step], canvas)
            R += r; P += p
        assert len(R) == len(imgs)
        return R, P

class HFDet(MicroBatch):
    kind, div = "hf", 1
    def __init__(self, name, repo, cls_name, revision=None, fixed_size=False, fixable=True, proc_cls="AutoImageProcessor"):
        self.name, self.fixed_size = name, fixed_size
        self.proc = getattr(transformers, proc_cls).from_pretrained(repo, revision=revision)
        self.model = load_hf_model(getattr(transformers, cls_name), repo, revision).to(DEVICE).eval()
        self.lab2cat = names_to_cat(self.model.config.id2label, name)
        self.accepts_mask = "pixel_mask" in inspect.signature(self.model.forward).parameters
        self.fix = Fix(); self.fix.SELF = None
        if fixable and not fixed_size:
            install_fix_hf(self.model, self.fix)

    # --- hooks that model-specific subclasses override ---
    def _improc(self):
        return self.proc
    def _extra(self, n):
        return {}
    def _post(self, out, imgs, kw):
        sizes = torch.tensor([[im.size[1], im.size[0]] for im in imgs])
        res = self.proc.post_process_object_detection(out, threshold=0.0, target_sizes=sizes)
        return [_pack(r["boxes"], r["scores"], r["labels"], self.lab2cat) for r in res]

    def _single(self, img):
        enc = self._improc()(images=[img], return_tensors="pt")
        pv = enc["pixel_values"][0]
        m = enc["pixel_mask"][0] if "pixel_mask" in enc else torch.ones(pv.shape[-2:], dtype=torch.long)
        return pv, m

    def input_size(self, img):
        return tuple(self._single(img)[0].shape[-2:])

    @torch.inference_mode()
    def _run(self, imgs, canvas=None):
        if canvas is None:
            enc = self._improc()(images=imgs, return_tensors="pt")
            pv = enc["pixel_values"]
            mask = enc["pixel_mask"] if "pixel_mask" in enc else None
        else:
            H, W = canvas; pvs, ms = [], []
            for im in imgs:
                p, m = self._single(im); h, w = p.shape[-2:]
                assert h <= H and w <= W, f"canvas {canvas} smaller than image {(h, w)}"
                pvs.append(F.pad(p, (0, W - w, 0, H - h))); ms.append(F.pad(m, (0, W - w, 0, H - h)))
            pv, mask = torch.stack(pvs), torch.stack(ms)
        self._last_mask = mask
        kw = {"pixel_values": pv.to(DEVICE)}
        if mask is not None and self.accepts_mask:
            kw["pixel_mask"] = mask.to(DEVICE)
            if self.fix.any_on():
                self.fix.begin([(int(mask[i, :, 0].sum()), int(mask[i, 0, :].sum())) for i in range(len(imgs))], pv.shape[-2:])
        kw.update(self._extra(len(imgs)))
        try:
            out = self.model(**kw)
        finally:
            self.fix.end()
        pad = (1 - mask.float().mean(dim=(1, 2))).tolist() if mask is not None else [0.0] * len(imgs)
        return self._post(out, imgs, kw), pad


class YolosDet(HFDet):
    """YOLOS has no padding mask. `rescale=True` tests the coordinate-frame hypothesis: if YOLOS predicts boxes
    relative to the padded canvas instead of the image, multiplying them by canvas/image size puts them back."""
    rescale = False
    def __init__(self, name, repo):
        super().__init__(name, repo, "YolosForObjectDetection", fixable=False)
    def _post(self, out, imgs, kw):
        if self.rescale and self._last_mask is not None:
            m = self._last_mask
            H, W = m.shape[-2:]
            h = m[:, :, 0].sum(1).float(); w = m[:, 0, :].sum(1).float()
            s = torch.stack([W / w, H / h, W / w, H / h], dim=1).to(out.pred_boxes.device)
            out.pred_boxes = out.pred_boxes * s[:, None, :]
        return super()._post(out, imgs, kw)

COCO_PROMPT = " . ".join(c["name"] for c in GT.loadCats(GT.getCatIds())) + " ."

class GroundingDINODet(HFDet):
    """Open-vocabulary detector prompted with all 80 COCO class names."""
    def __init__(self, name, repo):
        super().__init__(name, repo, "GroundingDinoForObjectDetection", fixable=False, proc_cls="AutoProcessor")
        self.lab2cat = {}
        n_tok = len(self.proc.tokenizer(COCO_PROMPT)["input_ids"])
        print(f"  prompt: {n_tok} tokens (model limit {getattr(self.model.config, 'max_text_len', '?')})")
    def _improc(self):
        return self.proc.image_processor
    def _extra(self, n):
        t = self.proc.tokenizer([COCO_PROMPT] * n, padding="longest", return_tensors="pt")
        return {k: v.to(DEVICE) for k, v in t.items()}
    def _post(self, out, imgs, kw):
        sizes = [(im.size[1], im.size[0]) for im in imgs]
        res = self.proc.post_process_grounded_object_detection(out, input_ids=kw["input_ids"], threshold=0.05,
                                                              text_threshold=0.25, target_sizes=sizes)
        packed = []
        for r in res:
            names = r.get("text_labels", r.get("labels"))
            cats = [NAME2CAT.get(_norm(n)) for n in names]
            keep = np.array([c is not None for c in cats], bool)
            packed.append({"boxes": r["boxes"].float().cpu().numpy()[keep], "scores": r["scores"].float().cpu().numpy()[keep],
                           "labels": np.array([c for c in cats if c is not None], dtype=int)})
        return packed

class Mask2FormerDet(HFDet):
    """Instance segmentation; boxes are taken from the predicted masks."""
    def __init__(self, name, repo):
        super().__init__(name, repo, "Mask2FormerForUniversalSegmentation", fixable=False)
    def _post(self, out, imgs, kw):
        sizes = [(im.size[1], im.size[0]) for im in imgs]
        res = self.proc.post_process_instance_segmentation(out, threshold=0.1, target_sizes=sizes, return_binary_maps=True)
        packed = []
        for r in res:
            seg = r["segmentation"]; info = r["segments_info"]
            boxes, scores, labels = [], [], []
            if seg is not None and len(info):
                segs = seg.bool().cpu().numpy() if seg.dim() == 3 else None
                for k, s in enumerate(info):
                    m = segs[k] if segs is not None else (seg == s["id"]).cpu().numpy()
                    c = self.lab2cat.get(int(s["label_id"]))
                    if c is None or not m.any():
                        continue
                    ys, xs = np.where(m)
                    boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1]); scores.append(s["score"]); labels.append(c)
            packed.append({"boxes": np.array(boxes, np.float32).reshape(-1, 4), "scores": np.array(scores, np.float32),
                           "labels": np.array(labels, int)})
        return packed

class TVDet(MicroBatch):
    kind, div, fixed_size = "tv", 32, False
    with_masks = False
    def __init__(self, name, builder, weights):
        self.name = name
        self.model = builder(weights=weights).to(DEVICE).eval()
        self.tf = weights.transforms()
        self.lab2cat = names_to_cat(dict(enumerate(weights.meta["categories"])), name)
        self._setup()

    def _setup(self):
        self._canvas = None
        self.fix = Fix()
        orig = self.model.transform.batch_images
        def batch_images(images, size_divisible=32):
            b = orig(images, size_divisible)
            H, W = self._canvas if self._canvas is not None else tuple(b.shape[-2:])
            assert b.shape[-2] <= H and b.shape[-1] <= W, f"canvas {(H, W)} smaller than batch {tuple(b.shape[-2:])}"
            if (H, W) == tuple(b.shape[-2:]):
                return b
            return F.pad(b, (0, W - b.shape[-1], 0, H - b.shape[-2]))
        self.model.transform.batch_images = batch_images
        install_fix_tv(self.model, self.fix)

    def _pack_masks(self, o):
        lab = o["labels"].cpu().numpy(); sc = o["scores"].float().cpu().numpy()
        keep = np.array([l in self.lab2cat for l in lab], bool) & (sc >= 0.05)
        ms = (o["masks"][:, 0] > 0.5).cpu().numpy()[keep]
        rles = []
        for m in ms:
            r = mask_utils.encode(np.asfortranarray(m.astype(np.uint8))); r["counts"] = r["counts"].decode("ascii"); rles.append(r)
        return {"boxes": o["boxes"].float().cpu().numpy()[keep], "scores": sc[keep],
                "labels": np.array([self.lab2cat[l] for l in lab[keep]], dtype=int), "masks": rles}

    @torch.inference_mode()
    def input_size(self, img):
        il, _ = self.model.transform([img if torch.is_tensor(img) else self.tf(img).to(DEVICE)])
        return tuple(il.image_sizes[0])

    @torch.inference_mode()
    def _run(self, imgs, canvas=None):
        self._canvas = canvas
        try:
            x = [im if torch.is_tensor(im) else self.tf(im).to(DEVICE) for im in imgs]
            il, _ = self.model.transform(x)
            H, W = il.tensors.shape[-2:]
            pad = [1 - (h * w) / (H * W) for h, w in il.image_sizes]
            if self.fix.any_on():
                d = self.model.transform.size_divisible
                sizes = [tuple(s) for s in il.image_sizes]
                self.fix.begin([(roundup(h, d), roundup(w, d)) for h, w in sizes] if self.fix.own else sizes, (H, W))
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP and DEVICE == "cuda"):
                out = self.model(x)
        finally:
            self._canvas = None; self.fix.end()
        if self.with_masks:
            return [self._pack_masks(o) for o in out], pad
        return [_pack(o["boxes"], o["scores"], o["labels"], self.lab2cat) for o in out], pad

def _tv(builder_name, weights_name, name):
    import torchvision.models.detection as D
    return lambda: TVDet(name, getattr(D, builder_name), getattr(D, weights_name).DEFAULT)

LOADERS = {
    # models from Exps 1-3 (used here for the mask and speed parts)
    "detr_r50": lambda: HFDet("detr_r50", "facebook/detr-resnet-50", "DetrForObjectDetection", revision="no_timm"),
    "frcnn_v2": _tv("fasterrcnn_resnet50_fpn_v2", "FasterRCNN_ResNet50_FPN_V2_Weights", "frcnn_v2"),
    "maskrcnn_v2": _tv("maskrcnn_resnet50_fpn_v2", "MaskRCNN_ResNet50_FPN_V2_Weights", "maskrcnn_v2"),
    "retinanet_v2": _tv("retinanet_resnet50_fpn_v2", "RetinaNet_ResNet50_FPN_V2_Weights", "retinanet_v2"),
    "deformable_detr": lambda: HFDet("deformable_detr", "SenseTime/deformable-detr", "DeformableDetrForObjectDetection", fixable=False),
    "fcos": _tv("fcos_resnet50_fpn", "FCOS_ResNet50_FPN_Weights", "fcos"),
    "rtdetr_r50": lambda: HFDet("rtdetr_r50", "PekingU/rtdetr_r50vd", "RTDetrForObjectDetection", fixed_size=True),
    # new in Exp 4
    "conditional_detr": lambda: HFDet("conditional_detr", "microsoft/conditional-detr-resnet-50", "ConditionalDetrForObjectDetection"),
    "dab_detr": lambda: HFDet("dab_detr", "IDEA-Research/dab-detr-resnet-50", "DabDetrForObjectDetection", fixable=False),
    "yolos_small": lambda: YolosDet("yolos_small", "hustvl/yolos-small"),
    "grounding_dino_tiny": lambda: GroundingDINODet("grounding_dino_tiny", "IDEA-Research/grounding-dino-tiny"),
    "mask2former_swin_s": lambda: Mask2FormerDet("mask2former_swin_s", "facebook/mask2former-swin-small-coco-instance"),
    "rtdetr_v2_r50": lambda: HFDet("rtdetr_v2_r50", "PekingU/rtdetr_v2_r50vd", "RTDetrV2ForObjectDetection", fixed_size=True),
}
# Published full COCO val2017 box AP; our 500-image batch-1 AP should be within about ±2.
REF_AP = {"deformable_detr": 44.5, "fcos": 39.2, "rtdetr_r50": 53.1, "detr_r50": 42.0, "frcnn_v2": 46.7, "maskrcnn_v2": 47.4, "retinanet_v2": 41.5,
          "conditional_detr": 41.0, "dab_detr": 42.2, "yolos_small": 36.1, "grounding_dino_tiny": 48.4,
          "mask2former_swin_s": None, "rtdetr_v2_r50": 53.4}

def load(name):
    free_mem()
    t0 = time.time()
    try:
        det = LOADERS[name]()
        det.micro = CFG["micro_batch"].get(name)
    except Exception as ex:
        print(f"✗ {name} failed to load: {type(ex).__name__}: {str(ex)[:300]}  — skipping this model")
        return None
    note = f"; runs in pieces of {det.micro} (padded to the full batch's canvas)" if det.micro else ""
    print(f"\n== {name} loaded in {time.time() - t0:.0f}s; mapped classes {len(set(det.lab2cat.values()))}/80{note}")
    return det

def free_mem():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

In [ ]:
import logging
logging.getLogger("ultralytics").setLevel(logging.ERROR)   # keep the log readable (fp32 is Ultralytics' default)

class YOLODet:
    """Ultralytics YOLO through its standard predict() API. `rect=True` is Ultralytics' default."""
    kind, div, fixed_size = "yolo", 1, True
    def __init__(self, name, weights):
        from ultralytics import YOLO
        self.name, self.model, self.rect, self.micro = name, YOLO(weights), True, None
        self.lab2cat = names_to_cat(self.model.names, name)
        self.fix = Fix(); self.fix.SELF = None
    def input_size(self, img):
        return (img.size[1], img.size[0])
    @torch.inference_mode()
    def run(self, imgs, canvas=None):
        arrs = [np.ascontiguousarray(np.asarray(im.convert("RGB"))[:, :, ::-1]) for im in imgs]   # BGR, like cv2
        res = self.model.predict(arrs, imgsz=CFG["yolo_imgsz"], conf=CFG["yolo_conf"], iou=CFG["yolo_iou"],
                                 max_det=100, rect=self.rect, verbose=False,
                                 device=0 if DEVICE == "cuda" else "cpu")
        out = []
        for r in res:
            b = r.boxes
            out.append(_pack(b.xyxy, b.conf, b.cls.int(), self.lab2cat))
        return out, [0.0] * len(imgs)

for _n in ["yolo11s", "yolo11m", "yolo26s", "yolo26m"]:
    LOADERS[_n] = (lambda n: (lambda: YOLODet(n, n + ".pt")))(_n)
LOADERS["rtdetr_r50"] = lambda: HFDet("rtdetr_r50", "PekingU/rtdetr_r50vd", "RTDetrForObjectDetection", fixed_size=True)
LOADERS["fcos"] = _tv("fcos_resnet50_fpn", "FCOS_ResNet50_FPN_Weights", "fcos")

In [ ]:
# Keypoint R-CNN: same structure as Faster R-CNN plus a keypoint head; return keypoints with the boxes
class TVPose(TVDet):
    def __init__(self):
        import torchvision.models.detection as D
        super().__init__("keypointrcnn", D.keypointrcnn_resnet50_fpn, D.KeypointRCNN_ResNet50_FPN_Weights.DEFAULT)
    @torch.inference_mode()
    def _run(self, imgs, canvas=None):
        self._kp = True
        return super()._run(imgs, canvas)

_orig_pack = _pack
def _pack_with_kp(boxes, scores, labels, lab2cat, kps=None):
    r = _orig_pack(boxes, scores, labels, lab2cat)
    if kps is not None:
        lab = labels.cpu().numpy(); keep = np.array([l in lab2cat for l in lab], bool)
        r["kpts"] = kps.float().cpu().numpy()[keep]
    return r

_TV_run = TVDet._run
@torch.inference_mode()
def _tv_run_kp(self, imgs, canvas=None):
    if not isinstance(self, TVPose):
        return _TV_run(self, imgs, canvas)
    # same as TVDet._run, but keeps the keypoints
    self._canvas = canvas
    try:
        x = [im if torch.is_tensor(im) else self.tf(im).to(DEVICE) for im in imgs]
        il, _ = self.model.transform(x)
        H, W = il.tensors.shape[-2:]
        pad = [1 - (h * w) / (H * W) for h, w in il.image_sizes]
        if self.fix.any_on():
            d = self.model.transform.size_divisible
            sizes = [tuple(s) for s in il.image_sizes]
            self.fix.begin([(roundup(h, d), roundup(w, d)) for h, w in sizes] if self.fix.own else sizes, (H, W))
        out = self.model(x)
    finally:
        self._canvas = None; self.fix.end()
    return [_pack_with_kp(o["boxes"], o["scores"], o["labels"], self.lab2cat, o["keypoints"]) for o in out], pad
TVDet._run = _tv_run_kp
LOADERS["keypointrcnn"] = lambda: TVPose()

## 5. Metrics and helpers

In [ ]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def greedy_match(p, q, thr, iou_min=0.5):
    ia = np.where(p["scores"] >= thr)[0]; ib = np.where(q["scores"] >= thr)[0]
    ia = ia[np.argsort(-p["scores"][ia])]
    M = iou_matrix(p["boxes"][ia], q["boxes"][ib]); used = set(); pairs = []
    for r, i in enumerate(ia):
        best, bc = -1.0, None
        for c, j in enumerate(ib):
            if c not in used and p["labels"][i] == q["labels"][j] and M[r, c] > best:
                best, bc = M[r, c], c
        if bc is not None and best >= iou_min:
            used.add(bc); pairs.append((i, ib[bc], best))
    return pairs, ia, ib

def compare(ref, new):
    """Image-level flip + detection-level change counts + max score change, ref vs new."""
    pairs, ia, ib = greedy_match(ref, new, CFG["flip_score_thr"])
    stable = sum(iou >= CFG["flip_iou"] for _, _, iou in pairs)
    moved = len(pairs) - stable
    lost, gained = len(ia) - len(pairs), len(ib) - len(pairs)
    allp, _, _ = greedy_match(ref, new, 0.05)
    dscore = max([abs(ref["scores"][i] - new["scores"][j]) for i, j, _ in allp], default=0.0)
    return {"flip": (moved + lost + gained) > 0, "stable": stable, "moved": moved, "lost": lost, "gained": gained,
            "max_score_delta": dscore}

def det_survival(ref, new, W, H):
    """For each confident ref detection: did it survive unchanged? plus its distance to the right/bottom edge."""
    pairs, ia, _ = greedy_match(ref, new, CFG["flip_score_thr"])
    ok = {i for i, _, iou in pairs if iou >= CFG["flip_iou"]}
    rows = []
    for i in ia:
        x1, y1, x2, y2 = ref["boxes"][i]
        rows.append({"changed": i not in ok, "d_right": (W - x2) / W, "d_bottom": (H - y2) / H,
                     "rel_area": (x2 - x1) * (y2 - y1) / (W * H), "score": ref["scores"][i]})
    return rows

def coco_ap(preds, idx=None):
    idx = list(range(N)) if idx is None else idx
    dets = []
    for k in idx:
        r = preds[k]; order = np.argsort(-r["scores"])[:100]
        for t in order:
            x1, y1, x2, y2 = r["boxes"][t]
            dets.append({"image_id": IDS[k], "category_id": int(r["labels"][t]),
                         "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(r["scores"][t])})
    if not dets:
        return 0.0
    with contextlib.redirect_stdout(io.StringIO()):
        dt = GT.loadRes(dets); E = COCOeval(GT, dt, "bbox"); E.params.imgIds = [IDS[k] for k in idx]
        E.evaluate(); E.accumulate(); E.summarize()
    return 100 * E.stats[0]

def run_batches(det, batches, canvas_fn=None, targets=None):
    """batches: list of index lists. canvas_fn(batch)->(H,W) or None. targets: optional list of index lists
    saying which members of each batch to record (others are companions). Returns preds, pads, seconds."""
    P, PD = [None] * N, [None] * N
    sync(); t0 = time.time()
    for bi, b in enumerate(batches):
        r, p = det.run([IMGS[i] for i in b], canvas=canvas_fn(b) if canvas_fn else None)
        keep = set(targets[bi]) if targets else set(b)
        for i, ri, pi in zip(b, r, p):
            if i in keep and P[i] is None:
                P[i], PD[i] = ri, pi
    sync()
    return P, PD, time.time() - t0

In [ ]:
B = CFG["batch_size"]
def chunks(order, k=B):
    order = list(order)
    return [order[i:i + k] for i in range(0, len(order), k)]

def run_on(det, batches, source, targets=None):
    """like run_batches, but images come from `source` (PIL images or GPU tensors)"""
    P, PD = [None] * N, [None] * N
    sync(); t0 = time.time()
    for bi, b in enumerate(batches):
        r, p = det.run([source[i] for i in b])
        keep = set(targets[bi]) if targets else set(b)
        for i, ri, pi in zip(b, r, p):
            if i in keep and P[i] is None:
                P[i], PD[i] = ri, pi
    sync()
    return P, PD, time.time() - t0

def warmup(det, source=None):
    source = IMGS if source is None else source
    for _ in range(2):
        det.run(source[:B]); det.run(source[:1])
    sync()

def with_thr(model, thr=None):
    CFG["flip_score_thr"] = thr if thr is not None else CFG["flip_thr_per_model"].get(model, 0.5)

def mask_compare(ref, new):
    c = compare(ref, new)
    pairs, _, _ = greedy_match(ref, new, CFG["flip_score_thr"])
    ious = [float(mask_utils.iou([new["masks"][j]], [ref["masks"][i]], [0])[0][0]) for i, j, _ in pairs]
    return {**c, "mask_flip": bool(c["flip"] or any(v < CFG["flip_iou"] for v in ious)), "min_mask_iou": min(ious, default=1.0)}

def coco_segm_ap(preds):
    dets = []
    for k, img_id in enumerate(IDS):
        r = preds[k]
        for t in np.argsort(-r["scores"])[:100]:
            dets.append({"image_id": img_id, "category_id": int(r["labels"][t]), "segmentation": r["masks"][t],
                         "score": float(r["scores"][t])})
    with contextlib.redirect_stdout(io.StringIO()):
        dt = GT.loadRes(dets); E = COCOeval(GT, dt, "segm"); E.params.imgIds = IDS
        E.evaluate(); E.accumulate(); E.summarize()
    return 100 * E.stats[0]

In [ ]:
def use_dataset(name):
    global GT, IDS, INFO, IMGS, N, bs1, rand0
    d = DATASETS[name]; GT, IDS, INFO, IMGS = d["GT"], d["IDS"], d["INFO"], d["IMGS"]; N = len(IDS)
    bs1 = [[i] for i in range(N)]; rand0 = chunks(np.random.default_rng(0).permutation(N))
    print(f"-> using {name}: {N} images")

def buckets_for(det):
    groups = {}
    for i in range(N):
        key = (INFO[i]["height"], INFO[i]["width"]) if det.kind == "yolo" else \
              tuple(roundup(s, det.div) for s in det.input_size(IMGS[i]))
        groups.setdefault(key, []).append(i)
    return [b for g in groups.values() for b in chunks(g)], len(groups)

def pose_compare(ref, new):
    """box-level comparison plus: does any matched confident person's pose move by > pose_change?"""
    c = compare(ref, new)
    pairs, _, _ = greedy_match(ref, new, CFG["flip_score_thr"])
    worst = 0.0
    for i, j, _ in pairs:
        x1, y1, x2, y2 = ref["boxes"][i]; s = max(np.sqrt(max((x2 - x1) * (y2 - y1), 1.0)), 1.0)
        a, b = ref["kpts"][i][:, :2], new["kpts"][j][:, :2]
        worst = max(worst, float(np.linalg.norm(a - b, axis=1).mean() / s))
    return {**c, "pose_flip": bool(c["flip"] or worst > CFG["pose_change"]), "pose_shift": worst}

def kp_ap(preds):
    dets = []
    for k, img_id in enumerate(IDS):
        r = preds[k]
        for t in np.argsort(-r["scores"])[:20]:
            kp = r["kpts"][t]; flat = []
            for x, y, v in kp:
                flat += [float(x), float(y), 1.0]
            dets.append({"image_id": img_id, "category_id": 1, "keypoints": flat, "score": float(r["scores"][t])})
    if not dets:
        return 0.0
    with contextlib.redirect_stdout(io.StringIO()):
        dt = KP_GT.loadRes(dets); E = COCOeval(KP_GT, dt, "keypoints"); E.params.imgIds = IDS
        E.evaluate(); E.accumulate(); E.summarize()
    return 100 * E.stats[0]

## 6. Part A — batch-size sweep (1 → 32)

In [ ]:
if RUN_PARTS["A"]:
    use_dataset("coco1k")
    AR = read_csv(f"{OUT}/A_sweep.csv").to_dict("records")
    done = {r["model"] for r in AR}
    for mname in CFG["sweep_models"]:
        if mname in done:
            print(f"== {mname}: already done, skipping"); continue
        det = None; free_mem()
        det = load(mname)
        if det is None:
            continue
        warmup(det)
        ref, PD1, sec1 = run_on(det, bs1, IMGS)
        AR.append({"model": mname, "variant": "none", "batch": 1, "images_changed": 0.0, "padded_share": float(np.mean(np.array(PD1) > 1e-6)),
                   "changed_among_padded": np.nan, "AP": coco_ap(ref), "img_per_s": N / sec1, "gpu": ENV["gpu"]})
        runs = [("none", b) for b in CFG["sweep_batches"]] + ([("self", 32)] if det.fix.SELF else [])
        for v, b in runs:
            det.fix.set(det.fix.SELF if v == "self" else ())
            P, PD, sec = run_on(det, chunks(np.random.default_rng(0).permutation(N), b), IMGS)
            det.fix.set(())
            flips = np.array([compare(ref[i], P[i])["flip"] for i in range(N)])
            # padded = the image received more padding than at batch 1 (for YOLO: its batch mixed several shapes)
            if det.kind == "yolo":
                padded = np.zeros(N, bool)
                for bt in chunks(np.random.default_rng(0).permutation(N), b):
                    if len({(INFO[i]["height"], INFO[i]["width"]) for i in bt}) > 1:
                        padded[bt] = True
            else:
                padded = np.array(PD) > np.array(PD1) + 1e-6
            AR.append({"model": mname, "variant": v, "batch": b, "images_changed": flips.mean(),
                       "padded_share": padded.mean(), "changed_among_padded": flips[padded].mean() if padded.any() else np.nan,
                       "AP": coco_ap(P), "img_per_s": N / sec, "gpu": ENV["gpu"]})
            r = AR[-1]
            print(f"  {mname:9s} {v:5s} bs={b:2d}: {100 * r['images_changed']:5.1f}% changed | padded {100 * r['padded_share']:5.1f}% "
                  f"| changed among padded {100 * r['changed_among_padded']:5.1f}% | AP {r['AP']:.2f} | {r['img_per_s']:.1f} img/s")
        det = None; free_mem()
        pd.DataFrame(AR).to_csv(f"{OUT}/A_sweep.csv", index=False)

AS = read_csv(f"{OUT}/A_sweep.csv")
if not AS.empty:
    display(AS.style.format({"images_changed": "{:.1%}", "padded_share": "{:.1%}", "changed_among_padded": "{:.1%}",
                             "AP": "{:.2f}", "img_per_s": "{:.1f}"}, na_rep="—"))
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    for m in AS.model.unique():
        d = AS[(AS.model == m) & (AS.variant == "none")].sort_values("batch")
        axes[0].plot(d.batch, d.images_changed * 100, "o-", label=m)
        axes[1].plot(d.batch, d.changed_among_padded * 100, "o-", label=m)
    for ax, t in [(axes[0], "% images changed"), (axes[1], "% changed among padded images")]:
        ax.set_xscale("log", base=2); ax.set_xticks(CFG["sweep_batches"] + [1]); ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
        ax.set_xlabel("batch size"); ax.set_ylabel(t); ax.legend(); ax.set_ylim(0, 100)
    plt.tight_layout(); plt.savefig(f"{OUT}/A_sweep.png", dpi=180); plt.show()

## 7. Part B — pose estimation (Keypoint R-CNN)

In [ ]:
if RUN_PARTS["B"] and not os.path.exists(f"{OUT}/B_pose.csv"):
    use_dataset("coco")
    det = None; free_mem()
    det = load(CFG["pose_model"]); warmup(det)
    bk, ncan = buckets_for(det)
    print(f"  exact-shape buckets: {ncan} canvases, {len(bk)} batches")
    preds, rows, imgs_rows = {}, [], []
    for v, b, batches in [("none", "bs1", bs1), ("none", "bs8_rand", rand0), ("self", "bs8_rand", rand0), ("none", "buckets", bk)]:
        det.fix.set(det.fix.SELF if v == "self" else ())
        P, PD, sec = run_on(det, batches, IMGS)
        preds[(v, b)] = P
        rows.append({"variant": v, "batching": b, "box_AP": coco_ap(P), "kp_AP": kp_ap(P), "img_per_s": N / sec, "gpu": ENV["gpu"]})
        print(f"  {v:5s} {b:9s} box AP {rows[-1]['box_AP']:.2f}  keypoint AP {rows[-1]['kp_AP']:.2f}  {N / sec:.1f} img/s")
    det.fix.set(())
    for (v, b), P in preds.items():
        if (v, b) != ("none", "bs1"):
            for i in range(N):
                imgs_rows.append({"variant": v, "batching": b, "image_id": IDS[i], **pose_compare(preds[("none", "bs1")][i], P[i])})
    det = None; preds = None; free_mem()
    pd.DataFrame(rows).to_csv(f"{OUT}/B_pose.csv", index=False); pd.DataFrame(imgs_rows).to_csv(f"{OUT}/B_pose_per_image.csv", index=False)

BP = read_csv(f"{OUT}/B_pose.csv"); BPI = read_csv(f"{OUT}/B_pose_per_image.csv")
if not BP.empty:
    b0 = BP[(BP.variant == "none") & (BP.batching == "bs1")].iloc[0]
    BP["d_box_AP"] = BP.box_AP - b0.box_AP; BP["d_kp_AP"] = BP.kp_AP - b0.kp_AP
    f = BPI.groupby(["variant", "batching"]).agg(images_changed=("flip", "mean"), poses_changed=("pose_flip", "mean"),
                                                 worst_pose_shift=("pose_shift", "max")).reset_index()
    TP = BP.merge(f, on=["variant", "batching"], how="left")
    display(TP.style.format({"box_AP": "{:.2f}", "kp_AP": "{:.2f}", "d_box_AP": "{:+.3f}", "d_kp_AP": "{:+.3f}",
                             "images_changed": "{:.2%}", "poses_changed": "{:.2%}", "worst_pose_shift": "{:.3f}", "img_per_s": "{:.1f}"}, na_rep="—"))

## 8. Part C — the same runs on the CPU

In [ ]:
if RUN_PARTS["C"] and not os.path.exists(f"{OUT}/C_cpu.csv"):
    use_dataset("cpu200")
    rows, preds = [], {}
    for dev in ["cuda", "cpu"]:
        if dev == "cuda" and not torch.cuda.is_available():
            continue
        DEVICE = dev
        for mname in CFG["cpu_models"]:
            det = None; free_mem()
            det = load(mname)
            if det is None:
                continue
            plan = [("none", "bs1", bs1), ("none", "bs8_rand", rand0)] + ([("self", "bs8_rand", rand0)] if det.fix.SELF else [])
            for v, b, batches in plan:
                det.fix.set(det.fix.SELF if v == "self" else ())
                t0 = time.time(); P, PD, sec = run_on(det, batches, IMGS)
                preds[(dev, mname, v, b)] = P
                print(f"  {dev:4s} {mname:9s} {v:5s} {b:9s} {N / sec:6.2f} img/s")
            det.fix.set(()); det = None; free_mem()
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    for (dev, m, v, b), P in preds.items():
        if (v, b) == ("none", "bs1"):
            continue
        ref = preds[(dev, m, "none", "bs1")]
        rows.append({"device": dev, "model": m, "variant": v, "batching": b,
                     "images_changed": float(np.mean([compare(ref[i], P[i])["flip"] for i in range(N)]))})
    for m in CFG["cpu_models"]:
        if ("cuda", m, "none", "bs1") in preds and ("cpu", m, "none", "bs1") in preds:
            a, c = preds[("cuda", m, "none", "bs1")], preds[("cpu", m, "none", "bs1")]
            rows.append({"device": "cpu vs cuda", "model": m, "variant": "none", "batching": "bs1",
                         "images_changed": float(np.mean([compare(a[i], c[i])["flip"] for i in range(N)]))})
    pd.DataFrame(rows).to_csv(f"{OUT}/C_cpu.csv", index=False)

CC = read_csv(f"{OUT}/C_cpu.csv")
if not CC.empty:
    print("Images changed vs batch 1 on the same device ('cpu vs cuda': batch-1 outputs compared across devices)")
    display(CC.pivot_table(index=["model", "variant", "batching"], columns="device", values="images_changed").style.format("{:.2%}", na_rep="—"))

## 9. Claims check

In [ ]:
K = CFG["claims"]; v = {}
try:
    ok_all, parts_ = True, []
    for m in AS.model.unique():
        d = AS[(AS.model == m) & (AS.variant == "none") & (AS.batch >= 2)].sort_values("batch")
        grow = bool(np.all(np.diff(d.images_changed.values) >= -0.01))
        band = float(d.changed_among_padded.max() - d.changed_among_padded.min())
        fx = AS[(AS.model == m) & (AS.variant == "self")]
        fx_ok = fx.empty or fx.images_changed.iloc[0] <= K["R1_fix"]
        ok_all &= grow and band <= 2 * K["R1_band"] and fx_ok
        parts_.append(f"{m}: {', '.join(f'b{int(b)} {100 * x:.1f}%' for b, x in zip(d.batch, d.images_changed))}; "
                      f"among padded {100 * d.changed_among_padded.min():.1f}-{100 * d.changed_among_padded.max():.1f}%"
                      + ("" if fx.empty else f"; fix@32 {100 * fx.images_changed.iloc[0]:.2f}%"))
    v["R1_batch_size"] = ("✓  " if ok_all else "✗  ") + " | ".join(parts_)
except Exception as ex:
    v["R1_batch_size"] = f"not complete ({type(ex).__name__})"
try:
    n = TP[(TP.variant == "none") & (TP.batching == "bs8_rand")].iloc[0]; s = TP[TP.variant == "self"].iloc[0]
    ok = n.poses_changed >= K["R2_normal"] and s.poses_changed <= K["R2_fix"] and abs(s.d_kp_AP) <= K["R2_dap"]
    v["R2_pose"] = (f"{'✓' if ok else '✗'}  normal: {100 * n.images_changed:.1f}% images / {100 * n.poses_changed:.1f}% poses changed, "
                    f"Δkp-AP {n.d_kp_AP:+.2f} | fix: {100 * s.poses_changed:.2f}% poses, Δkp-AP {s.d_kp_AP:+.3f}")
except Exception as ex:
    v["R2_pose"] = f"not complete ({type(ex).__name__})"
try:
    parts_, ok = [], True
    for m in CC.model.unique():
        g = lambda d, var: CC[(CC.model == m) & (CC.device == d) & (CC.variant == var) & (CC.batching == "bs8_rand")].images_changed
        gpu, cpu = g("cuda", "none"), g("cpu", "none")
        if len(gpu) and len(cpu):
            ok &= abs(gpu.iloc[0] - cpu.iloc[0]) <= K["R3_tol"]
            fx = CC[(CC.model == m) & (CC.variant == "self")].images_changed
            if len(fx):
                ok &= fx.max() <= K["R3_fix"]
            x = CC[(CC.model == m) & (CC.device == "cpu vs cuda")].images_changed
            parts_.append(f"{m}: T4 {100 * gpu.iloc[0]:.1f}% vs CPU {100 * cpu.iloc[0]:.1f}%"
                          + (f", fix ≤ {100 * fx.max():.2f}%" if len(fx) else "")
                          + (f", batch-1 T4-vs-CPU {100 * x.iloc[0]:.1f}%" if len(x) else ""))
    v["R3_hardware"] = ("✓  " if ok and parts_ else "✗  ") + " | ".join(parts_)
except Exception as ex:
    v["R3_hardware"] = f"not complete ({type(ex).__name__})"
for k, x in v.items():
    print(f"{k:14s} {x}")
with open(f"{OUT}/manifest.json", "w") as f:
    json.dump({"env": ENV, "config": CFG, "verdicts": v}, f, indent=2, default=str)
print("saved:", sorted(os.listdir(OUT)))

## What to bring back

Upload `A_sweep.csv`, `B_pose.csv`, `B_pose_per_image.csv`, `C_cpu.csv` and `manifest.json` from the `exp7` folder.